# Example Progression 14 — LLM Group (Multi-Provider) (SDK, interactive)

Interactive counterpart of [`wf_examples/wf_example_progression_14.py`](../../wf_examples/wf_example_progression_14.py).
It imports the **exact same** `build_assistant_workflow()` from that script (no duplication),
creates the workflow on the dev server, drives a **capped** run, fetches the completed run, and
cleans up.

Nodes use an `LLMGroupConfig` racing multiple providers with patience timers (proactive / parallel-select-one modes).

> **Async-first.** Uses `AsyncWorkflowClient` with top-level `await` (the setup cell calls
> `nest_asyncio.apply()`). Every method also exists on the synchronous `WorkflowClient`.

In [ ]:
# This notebook lives one level below notebooks/, where _bootstrap.py lives. Walk up from the
# working dir to find it, add that dir to sys.path, then import it so `import interactly` works
# with no install needed. (Works whether the kernel cwd is notebooks/ or wf_example_notebooks/.)
import os, sys
from pathlib import Path
import nest_asyncio
from dotenv import load_dotenv

nest_asyncio.apply()

_notebooks_dir = None
for _cand in [Path.cwd(), *Path.cwd().parents]:
    if (_cand / '_bootstrap.py').exists():
        _notebooks_dir = _cand
        break
if _notebooks_dir and str(_notebooks_dir) not in sys.path:
    sys.path.insert(0, str(_notebooks_dir))

import _bootstrap  # noqa: F401 - enables `import interactly` (no install needed)

project_root = Path.cwd()
while project_root.parent != project_root:
    if (project_root / '.env').exists():
        break
    project_root = project_root.parent
env_path = project_root / '.env'
if env_path.exists():
    load_dotenv(dotenv_path=str(env_path))
    print(f'Loaded .env from: {env_path}')
else:
    print(f'Warning: .env file not found at {env_path}')

In [ ]:
# Interactly credentials are read from environment variables.
# Convenience defaults point at the dev "Workflow Illustrations" org; your shell env always wins.
os.environ.setdefault('INTERACTLY_BASE_URL', 'https://api-dev.interactly.ai/workflows')
os.environ.setdefault('INTERACTLY_TEAM_ID', '67458e762b7d3dc15aaea5b5')
os.environ.setdefault('INTERACTLY_USER_ID', '687b1a4f745c8e6806c98d91')

# The bearer token is a secret — never hardcode it in the notebook.
assert os.environ.get('INTERACTLY_API_KEY'), (
    'Set INTERACTLY_API_KEY in your environment before running this notebook.'
)
print(f"TEAM ID is: {os.getenv('INTERACTLY_TEAM_ID')}")
print(f"BASE URL is: {os.getenv('INTERACTLY_BASE_URL')}")

In [ ]:
import nest_asyncio
from interactly import AsyncWorkflowClient

nest_asyncio.apply()
client = AsyncWorkflowClient()
print('Connected to', client._base_url)

## 1. Build the workflow

Import `build_assistant_workflow()` from the example script and call it. It returns the workflow config; we supply the dynamic variables the script uses at runtime.

In [ ]:
# Import the builder straight from the example script instead of duplicating it, so the
# notebook always runs the exact same workflow definition as wf_examples/wf_example_progression_14.py.
# Importing the module only defines functions; its interactive main()/run() is guarded by
# `if __name__ == "__main__"`, so nothing runs on import.
_wf_examples = None
for _cand in [Path.cwd(), *Path.cwd().parents]:
    if (_cand / 'wf_examples' / 'wf_example_progression_14.py').exists():
        _wf_examples = _cand / 'wf_examples'
        break
assert _wf_examples, 'Could not locate the wf_examples/ directory.'
if str(_wf_examples) not in sys.path:
    sys.path.insert(0, str(_wf_examples))

from wf_example_progression_14 import build_assistant_workflow

workflow_config_full = build_assistant_workflow()
# This example's build() returns only the workflow. Supply the same dynamic variables the
# script uses at runtime (also seeded into WorkflowConfig.miscellaneous.default_dynamic_variables).
dynamic_variables = {}
print('Built workflow:', workflow_config_full.workflow_config.name)
print('Nodes:', [x.name for x in workflow_config_full.node_configs])
print('Edges:', [x.name for x in workflow_config_full.edge_configs])
print('dynamic_variables:', dynamic_variables)

## 2. Create the workflow on the dev server

Upload with `aupload_and_get_handle()` (as the script does), which creates + activates a `v0`
version and returns an `AsyncWorkflowHandle` that tracks the `run_id` across turns.

In [ ]:
from interactly import aupload_and_get_handle
from interactly.runtime.handle import AsyncWorkflowHandle

handle: AsyncWorkflowHandle = await aupload_and_get_handle(
    client,
    workflow_config_full,
    dynamic_variables=dynamic_variables,
)
WF_ID = handle.workflow_id
print(f'Uploaded workflow id={WF_ID}')

## 3. Drive a capped multi-turn chat

The script reads turns from `input()` in an unbounded loop. Here we send a **fixed, capped**
list of user turns with the per-turn `arun()` pattern (first turn `START`, rest `DATA`).

In [ ]:
import asyncio

from langchain_core.messages import HumanMessage

from interactly import APIConnectionError, APIError
from interactly.configs import (
    LLMNodeRunInput,
    NodesRunInputs,
    WorkflowCommand,
    WorkflowRunInput,
)
from interactly.runtime.events import (
    AssistantResponseEvent,
    BusyWaitForUserMessageEvent,
    EndRunNodeEvent,
    EndWorkflowEvent,
    HttpRequestNodeEvent,
    WorkerLLMNodeEvent,
)

MAX_EVENTS_PER_TURN = 100  # guard against runaway streams

# The dev server is occasionally flaky: the /execute route can intermittently return a
# transient 404/5xx (or a dropped connection). Retry those a few times with backoff so a
# blip does not fail the whole notebook. Non-transient errors (e.g. 422 validation) are
# re-raised immediately.
TRANSIENT_STATUSES = {404, 408, 425, 429, 500, 502, 503, 504}
MAX_TURN_RETRIES = 6


async def send_message(user_text=None, *, command=WorkflowCommand.DATA):
    """Send one turn on thread \"0\" (or a bare kickoff if user_text is None) and print events.

    Retries on transient server/network errors (the run POST buffers all events, so a failure
    happens before any event is yielded — retrying re-sends the same turn cleanly).
    """
    if user_text is not None:
        print(f'User: {user_text}')
    kwargs = dict(command=command, dynamic_variables=dynamic_variables)
    if user_text is not None:
        kwargs['thread_to_node_inputs'] = {
            '0': NodesRunInputs(node_run_inputs=[LLMNodeRunInput(messages=[HumanMessage(content=user_text)])])
        }
    run_input = WorkflowRunInput(**kwargs)

    attempt = 0
    while True:
        events = []
        try:
            async for event in handle.arun(run_input):
                events.append(event)
                if isinstance(event, AssistantResponseEvent) and event.content:
                    print(f'  Assistant ({event.origin_node_name}): {event.content}')
                elif isinstance(event, BusyWaitForUserMessageEvent):
                    print(f'  (waiting for user at {event.origin_node_name})')
                elif isinstance(event, HttpRequestNodeEvent):
                    print(f'  (HTTP request at {event.origin_node_name}: status={getattr(event, "status_code", "?")})')
                elif isinstance(event, EndRunNodeEvent):
                    _tr = getattr(getattr(event, 'run_output', None), 'tool_result', None)
                    print(f'  (node run ended at {event.origin_node_name})' + (f' tool_result={_tr}' if _tr is not None else ''))
                elif isinstance(event, WorkerLLMNodeEvent):
                    print(f'  (worker LLM at {event.origin_node_name}: {event.type})')
                elif isinstance(event, EndWorkflowEvent):
                    print('  (workflow ended)')
                if len(events) > MAX_EVENTS_PER_TURN:
                    print('  ! stopping turn early (event cap reached)')
                    break
            return events
        except (APIConnectionError, APIError) as exc:
            status = getattr(exc, 'status_code', None)
            transient = isinstance(exc, APIConnectionError) or status in TRANSIENT_STATUSES
            if events or not transient or attempt >= MAX_TURN_RETRIES:
                raise
            attempt += 1
            wait = 1.5 * attempt
            print(f'  transient {type(exc).__name__} (status={status}); retry {attempt}/{MAX_TURN_RETRIES} in {wait:.1f}s...')
            await asyncio.sleep(wait)

In [ ]:
# A fixed, capped list of user turns (replaces the script's interactive input() loop).
USER_TURNS = [
    "What is the difference between a deductible and an out-of-pocket maximum?",
    "Can I use my HSA to pay for gym memberships?"
]

ended = False
try:
    for i, text in enumerate(USER_TURNS):
        print('=' * 80)
        print(f'--- Turn {i + 1}/{len(USER_TURNS)} ---')
        cmd = WorkflowCommand.START if i == 0 else WorkflowCommand.DATA
        turn_events = await send_message(text, command=cmd)
        if any(isinstance(e, EndWorkflowEvent) for e in turn_events):
            ended = True
            print('\nWorkflow reached its end node — stopping the chat.')
            break
except APIError as exc:
    # The workflow was created fine; the dev server hit a server-side error executing it.
    # Surface it (it may be transient — re-run this cell — or a node type not enabled on dev).
    print(f'\n⚠️  dev server returned {type(exc).__name__} (status={getattr(exc, "status_code", None)}) while executing this workflow.')
    print('    The workflow itself was created successfully; this is a server-side execution issue on the dev deployment.')

print(f'\nSession run_id: {handle.run_id}')
print(f'Workflow ended cleanly: {ended}')
RUN_ID = handle.run_id

## 4. Fetch the completed run

Fetch the run back by the `run_id` the handle tracked and print each input/output pair.

In [ ]:
from interactly import Run
from interactly.runtime.events import parse_event

if not RUN_ID:
    print('No run to fetch (the workflow did not start a run).')
else:
    run: Run = await client.runs.get(RUN_ID)
    print(f'Run {run.id}  status={run.status}  started={run.started_at}')

    def _get(obj, key):
        return getattr(obj, key, None) if not isinstance(obj, dict) else obj.get(key)

    for i, pair in enumerate(run.input_output_pairs):
        run_output = _get(pair, 'run_output')
        events = _get(run_output, 'events') or []
        assistant_texts = []
        for raw in events:
            try:
                ev = parse_event(raw)
            except Exception:
                continue
            if isinstance(ev, AssistantResponseEvent) and ev.content:
                assistant_texts.append(ev.content)
        print(f'  Pair {i}: {len(events)} event(s); assistant said: {assistant_texts}')

## 5. Cleanup

Delete the workflow (removing its versions and this run) so nothing is left on the dev server.

In [ ]:
await client.workflows.delete(WF_ID)
print(f'Workflow {WF_ID} deleted.')

await client.close()
print('Client closed.')